In [2]:
import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========================== 外部参数配置 ==========================
# 设备成本参数
C_arm = 15    # 大机器人单价（万元/台）
C_amr = 2     # 小机器人单价（万元/台）
C_old_mod = 5 # 旧机器人改造单价（万元/台）
C_rack = 0.2  # 单列货架成本（万元/列）
C_land = 0.1  # 场地成本（万元/㎡）
A_total = 5000 # 仓库总面积（㎡）
k = 100       # 单列货架占地面积（㎡/列）
MAX_COST = 1200  # 成本上限约束（万元）

# 效率参数
alpha = 60    # 旧机器人单位处理量（箱/小时）
beta = 0.2    # 面积衰减系数
v_vertical = 2   # 大机器人垂直速度（m/s）
H_avg = 15    # 平均存储高度（m）
mu = 1        # 操作效率系数
v_horizontal = 3 # 小机器人水平速度（m/s）
D_avg = 50    # 缓存区到工作站平均距离（m）
TaskRate = 200  # 单位时间料箱处理需求（箱/小时）
gamma = 0.1   # 缓存阻塞惩罚系数
manual_time_per_box = 6 / 3600  # 人工操作时间（小时）
charge_per_robot = 0.15  # 每台机器人充电时间占比[9](@ref)

# 新增参数
base_road_ratio = 0.25  # 基础通道面积占比[4](@ref)
road_per_robot = 0.005  # 每增加1台机器人需增加的通道比例

# 决策变量范围
arm_range = [1, 30]     # N_arm范围
amr_range = [1, 100]    # N_amr范围
old_range = [0, 50]     # N_old范围
eta_range = [0.0, 1.0]  # η_new范围

# ========================== 定义优化问题 ==========================
class WarehouseProblem(ElementwiseProblem):
    def __init__(self):
        # 决策变量：5个变量（3个整数+2个连续）
        super().__init__(
            n_var=5,
            n_obj=3,
            n_ieq_constr=6,  # 增加一个约束
            xl=np.array([arm_range[0], amr_range[0], old_range[0], 
                         eta_range[0], 0.2]),  # 新增道路比例变量
            xu=np.array([arm_range[1], amr_range[1], old_range[1], 
                         eta_range[1], 0.4]),  # 道路比例上限40%
            vtype=[int, int, int, float, float]
        )
    
    def _evaluate(self, x, out, *args, **kwargs):
        # 解析决策变量
        N_arm, N_amr, N_old, eta_new, road_ratio = x
        N_columns = 6 * N_arm  # 货架数量
        eta_old = 1 - eta_new
        
        # ---- 面积计算（含通道）[4](@ref) ----
        # 动态通道比例：基础比例 + 机器人数量影响
        total_robots = N_arm + N_amr
        effective_road_ratio = base_road_ratio + total_robots * road_per_robot
        effective_road_ratio = min(effective_road_ratio, 0.4)  # 上限40%
        
        A_used = N_columns * k * (1 + effective_road_ratio)  # 实际占用面积
        A_available = A_total - A_used  # 可用面积
        
        # ---- 约束条件计算 ----
        g1 = 3 * N_arm - N_amr  # N_amr >= 3*N_arm
        g2 = A_used - A_total    # 面积约束
        g3 = TaskRate - 2 * N_columns  # 容量约束
        g4 = effective_road_ratio - 0.4  # 通道比例上限
        g5 = A_available / A_total - 0.5  # 可用面积比例约束[4](@ref)
        
        # ---- 成本计算 ----
        cost = (
            C_arm * N_arm + 
            C_amr * N_amr +
            C_old_mod * N_old +
            C_rack * N_columns +
            C_land * A_used  # 按实际占用面积计算
        )
        
        # ---- 新增成本约束 ----
        g6 = cost - MAX_COST  # 成本上限
        
        # ---- 效率计算（含充电时间）[9](@ref) ----
        # 充电时间影响因子
        charge_factor = 1 - charge_per_robot * np.log1p(total_robots)
        
        # 旧系统效率
        OldSysEff = N_old * alpha * (1 - beta * A_used / A_total) * charge_factor
        
        # 新系统效率
        ArmEff = N_arm * (v_vertical / H_avg) * mu * 3600 * charge_factor
        AmrEff = N_amr * (v_horizontal / D_avg) * (A_used / A_total) * 3600 * charge_factor
        CacheBlockRate = max(0, (TaskRate - 2 * N_columns) / (2 * N_columns))
        NewSysEff = min(ArmEff, AmrEff) * (1 - gamma * CacheBlockRate)
        
        # 综合效率
        efficiency = eta_new * NewSysEff + eta_old * OldSysEff
        
        # 人工操作限制
        max_manual_capacity = 1 / manual_time_per_box
        efficiency = min(efficiency, max_manual_capacity)
        
        # ---- 总容量计算 ----
        capacity = 288 * N_columns + 27 * N_old
        
        # ---- 输出结果 ----
        out["F"] = [cost, -efficiency, -capacity]
        out["G"] = [g1, g2, g3, g4, g5, g6]  # 所有约束

# ========================== 优化执行 ==========================
problem = WarehouseProblem()

algorithm = NSGA2(
    pop_size=150,
    sampling=IntegerRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(eta=20),
    eliminate_duplicates=True
)

termination = get_termination("n_gen", 300)

res = minimize(
    problem,
    algorithm,
    termination,
    seed=1,
    verbose=True
)

# ========================== 结果分析与可视化 ==========================
# ...（可视化代码保持不变，参考原始实现）...

# 筛选符合约束的解
feasible_mask = (res.F[:, 0] <= MAX_COST) & (res.F[:, 1] <= 0) & (res.F[:, 2] <= 0)
feasible_F = res.F[feasible_mask]
feasible_pop = [res.pop[i] for i in range(len(res.pop)) if feasible_mask[i]]

# 分析道路比例与效率的关系
road_ratios = [ind.X[4] for ind in feasible_pop]
efficiencies = [-f[1] for f in feasible_F]

plt.figure(figsize=(10, 6))
plt.scatter(road_ratios, efficiencies, c=feasible_F[:, 0], cmap='viridis')
plt.colorbar(label='成本 (万元)')
plt.xlabel('通道面积比例')
plt.ylabel('系统效率 (箱/小时)')
plt.title('通道面积比例与系统效率关系')
plt.grid(alpha=0.3)
plt.savefig('road_ratio_vs_efficiency.png', dpi=300)
plt.show()

# 输出最优解分析
print(f"优化结果统计（共{len(feasible_pop)}个可行解）")
print(f"平均通道比例: {np.mean(road_ratios):.3f}")
print(f"最小效率: {np.min(efficiencies):.1f} 箱/小时")
print(f"最大效率: {np.max(efficiencies):.1f} 箱/小时")

# 提取Pareto最优解
pareto = np.ones(len(feasible_F), dtype=bool)
for i in range(len(feasible_F)):
    for j in range(len(feasible_F)):
        if i != j and np.all(feasible_F[j] <= feasible_F[i]) and np.any(feasible_F[j] < feasible_F[i]):
            pareto[i] = False
            break

pareto_front = feasible_F[pareto]
pareto_solutions = [feasible_pop[i] for i in range(len(feasible_pop)) if pareto[i]]

# 输出Pareto解
print("\nPareto最优解集:")
for i, sol in enumerate(pareto_solutions[:5]):
    x = sol.X
    print(f"解{i+1}: N_arm={x[0]}, N_amr={x[1]}, N_old={x[2]}, "
          f"η_new={x[3]:.2f}, 道路比例={x[4]:.3f}")

Exception: ('Problem Error: G can not be set, expected shape (150, 5) but provided (150, 6)', ValueError('cannot reshape array of size 900 into shape (150,5)'))